# Matrix Factorisation on Movie Lens 1M dataset
Dataset from: [Movie Lens 1M Dataset](https://grouplens.org/datasets/movielens/1m/)

In [ ]:
# Standard
import json
import os

# Third-party
import numpy as np
import pandas as pd

# Local
from Dataset.ml1m_data_loader import load_and_merge_data, ml_test_train_split
from evals.offline_eval_metrics import OfflineModelEvaluator, OfflineSlateEvaluator, coverage
from models.matrix_factorisation.NMF_matrix_factorisation import NMFMatrixFactorisation

### Load dataset

In [ ]:
df = load_and_merge_data()
df_train, df_test = ml_test_train_split(df, test_proportion=0.2)

In [ ]:
df_train.head()

In [ ]:
df_test.head()

### Matrix Factorisation
Matrix factorisation algorithm applied to Top-N and Similarity (by movie) slates.

i.e. answers the questions: "what are the top N movies for a specific user" and "because someone watched a movie, they should watch"


Using:
- [SKLearn NMF (Non-Negative Matrix Factorisation)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)]
- Similarity code from [here](https://github.com/dinesh-git17/movie_recommendation/tree/main)
- Top-N code from [here](https://medium.com/@quindaly/step-by-step-nmf-example-in-python-9974e38dc9f9)

In [ ]:
model_name = "NMF_MF"
NMF_model = NMFMatrixFactorisation(df_train, min_ratings=0, n_components=50)

### Evaluation metrics
Offline metrics - adapted from [here](https://github.com/aryan-jadon/Evaluation-Metrics-for-Recommendation-Systems/blob/main/recommenders/evaluation/python_evaluation.py)

Table from [here](https://github.com/recommenders-team/recommenders/blob/main/examples/03_evaluate/evaluation.ipynb)
|Metric|Range|Selection criteria|Limitation|Reference|
|------|-------------------------------|---------|----------|---------|
|RMSE|$> 0$|The smaller the better.|May be biased, and less explainable than MAE|[link](https://en.wikipedia.org/wiki/Root-mean-square_deviation)|
|MAE|$\geq 0$|The smaller the better.|Dependent on variable scale.|[link](https://en.wikipedia.org/wiki/Mean_absolute_error)|
|R2|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Coefficient_of_determination)|
|Explained variance|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Explained_variation)|

In [ ]:
model_eval = OfflineModelEvaluator()

In [ ]:
# Extract predicted ratings for every movie in df_train
pred_ratings_train = []
for row in df_train.itertuples(index=False):
    pred_ratings_train.append(NMF_model.V.loc[row.user_id][row.title])

In [ ]:
# Extract predicted ratings for every movie in df_test
pred_ratings_test = []
for row in df_test.itertuples(index=False):
    pred_ratings_test.append(NMF_model.V.loc[row.user_id][row.title])

In [ ]:
print("Metrics on training data:")
[print(f"{k}: {v:.2f}") for k, v in model_eval.calculate_metrics(list(df_train.rating), pred_ratings_train).items()]
print("\nMetrics on test data:")
[print(f"{k}: {v:.2f}") for k, v in model_eval.calculate_metrics(list(df_test.rating), pred_ratings_test).items()]

### Apply model

In [ ]:
recs = NMF_model.movie_similarity("101 Dalmatians (1961)")
recs

In [ ]:
recs = NMF_model.movie_similarity("101 Dalmatians (1996)")
recs

In [ ]:
recs = NMF_model.movie_similarity("10 Things I Hate About You (1999)")
recs

In [ ]:
recs = NMF_model.movie_similarity("Young Guns (1988)")
recs

Thoughts:
* Not recommending sequels
* Not using a test-train split -> this algorithm won't work if the requested movie doesn't exist in the pivot table
* Therefore, can't handle new movies or users

## Top-N movies for user

In [ ]:
user_id  = 44
#NMF_model.understand_user_profile(user_id)
rec = NMF_model.user_top_N(user_id)

In [ ]:
print(f"Recommendations for user {user_id}:")
rec_df = NMF_model.get_recommend_dataframe(rec)
display(rec_df)

In [ ]:
rec_ids = rec_df["movie_id"]
eval = OfflineSlateEvaluator(rec_ids, df_test, user_id, 3.5, "evals/movie_embeddings.pkl")

In [ ]:
offline_slate_metrics = eval.calculate_metrics(pred_ratings=rec_df["pred_ratings"], k=10)
for k, v in offline_slate_metrics.items():
    print(f"{k}: {v:.2f}")

### Evaluate across subset of users in test dataset
Note: depending on how fast your modell runs, you may want to save out the metrics as you go along

In [ ]:
# Params:
N = 10
K = 10
min_rating_for_relevance = 3.5
users_to_eval = df_test["user_id"][:500]

In [ ]:
# Convert numpy types to Python types for JSON serialization
def convert_numpy_types(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


In [ ]:
results = []
all_recommendations = []
print(f"Evaluating recommendations for {len(users_to_eval)} users")
for user_id in users_to_eval:
    # Generate recommendation slate
    rec = NMF_model.user_top_N(user_id, N)
    rec_df = NMF_model.get_recommend_dataframe(rec)
    rec_ids = rec_df["movie_id"]
    all_recommendations.append(rec_ids)

    # Run offline eval for this slate
    eval = OfflineSlateEvaluator(rec_ids, df_test, user_id, min_rating_for_relevance, "evals/movie_embeddings.pkl")
    results.append(eval.calculate_metrics(pred_ratings=rec_df["pred_ratings"], k=K))

# Calculate overall metrics
df_results = pd.DataFrame(results)
# Save results
output_dir = "evals/eval_results"
os.makedirs(output_dir, exist_ok=True) # Create output directory
df_results.to_csv(f'{output_dir}/user_metrics_{model_name}.csv', index=False)
overall_metrics = {
    'precision_at_k': df_results['precision_at_k'].mean(),
    'recall_at_k': df_results['recall_at_k'].mean(),
    'f1_at_k': df_results['f1_at_k'].mean(),
    'ndcg_at_k': df_results['ndcg_at_k'].mean(),
    'hit_rate_at_k': df_results['hit_rate_at_k'].mean(),
    'mean_average_precision': df_results['average_precision'].mean(),
    'mean_intra_list_similarity': df_results['intra_list_similarity'].mean(),
    'mean_gini_index': df_results['gini_index'].mean(),
    'catalog_coverage': coverage(all_recommendations, df_train['movie_id'].nunique()),
    'num_users': len(results),
    'k': K
}
with open(f'evals/eval_results/overall_metrics_{model_name}.json', 'w') as f:
    overall_metrics_serializable = {k: convert_numpy_types(v) for k, v in overall_metrics.items()}
    json.dump(overall_metrics, f, indent=2)

# Show output
for k, v in overall_metrics.items():
    print(f"{k}: {v:.2f}")

## User profile evaluation

|User|Total Ratings|Overall Avg Rating|Overall Std Dev|Must include| Should include| Must exclude|
|------|-------------------------------|---------|----------|---------|----|---|
| 6013 | 124 | 4.08 | 1.23 | Comedy, Drama | Musical, Romance | Action |
| 2195 | 258 | 3.41 | 1.41 | Action, Sci-Fi | Drama | Musical |
| 1198 | 102 | 3.66 | 1.51 | Action | Drama, Thriller | Children's |
| 3662 | 88  | 3.03 | 1.64 | Sci-Fi | Horror | Comedy |
| 4713 | 66  | 3.03 | 1.55 | Drama | Romance | Horror |


In [ ]:
user_ids = [6013, 2195, 1198, 3662, 4713]

In [ ]:
for user_id in user_ids:
    NMF_model.understand_user_profile(user_id, rating_dist=False, wc=False)
    rec = NMF_model.user_top_N(user_id)
    print(f"Recommendations for user {user_id}")
    display(NMF_model.get_recommend_dataframe(rec))

Based on the requirements outlined in the table above - the NMF MF model passes all qualitative criteria